In [0]:

# Tamaño de la información cargada.
lineas = spark.sql("""
    SELECT
        COUNT(*) AS filas,
        COUNT(DISTINCT source_file) AS archivos_con_datos
    FROM workspace.default.bronze_pos_raw
""").first()

# Resumen de pedidos e importes.
pedidos = spark.sql("""
    SELECT
        COUNT(*) AS pedidos,
        COUNT(DISTINCT store_number) AS tiendas,
        COUNT(DISTINCT fecha) AS fechas_con_datos,
        MIN(fecha) AS fecha_inicial,
        MAX(fecha) AS fecha_final,
        COUNT(importe_comparable) AS pedidos_con_importe,
        COUNT(*) - COUNT(importe_comparable) AS pedidos_sin_importe,
        ROUND(
            100.0 * COUNT(importe_comparable) / NULLIF(COUNT(*), 0),
            2
        ) AS cobertura_pct,
        ROUND(SUM(importe_comparable), 2) AS importe_estimado
    FROM workspace.default.silver_orders
""").first()

print("RESUMEN DE LA BASE COMPLETA")
print(f"Filas originales cargadas: {lineas['filas']:,}")
print(f"Archivos con datos: {lineas['archivos_con_datos']:,}")
print(f"Pedidos: {pedidos['pedidos']:,}")
print(f"Tiendas: {pedidos['tiendas']:,}")
print(f"Fechas con datos: {pedidos['fechas_con_datos']:,}")
print(f"Primera fecha: {pedidos['fecha_inicial']}")
print(f"Última fecha: {pedidos['fecha_final']}")
print(f"Pedidos con importe: {pedidos['pedidos_con_importe']:,}")
print(f"Pedidos sin importe: {pedidos['pedidos_sin_importe']:,}")
print(f"Cobertura del importe: {pedidos['cobertura_pct']}%")
print(f"Suma del importe estimado: {pedidos['importe_estimado']}")

In [0]:
mensual = spark.table("workspace.default.gold_monthly_kpis")

display(mensual.orderBy("mes"))

In [0]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

# Solo llevamos a pandas el resumen mensual.
datos = (
    spark.table("workspace.default.gold_monthly_kpis")
    .orderBy("mes")
    .toPandas()
)

datos["mes"] = pd.to_datetime(datos["mes"])

graficas = [
    (
        "pedidos_por_tienda_dia",
        "Pedidos por tienda-día con actividad",
        "Pedidos",
        "#2563EB"
    ),
    (
        "importe_promedio_por_pedido",
        "Importe promedio de pedidos con estimación",
        "Importe",
        "#059669"
    ),
    (
        "pedidos_con_descuento_pct",
        "Pedidos con descuento informado",
        "Porcentaje de pedidos",
        "#D97706"
    ),
    (
        "cobertura_importe_pct",
        "Pedidos con importe comparable",
        "Porcentaje de pedidos",
        "#7C3AED"
    )
]

fig, ejes = plt.subplots(2, 2, figsize=(15, 9))

for eje, (columna, titulo, etiqueta, color) in zip(
    ejes.flat, graficas
):
    valores = datos[columna].astype(float)

    eje.plot(
        datos["mes"],
        valores,
        marker="o",
        markersize=4,
        linewidth=2,
        color=color
    )

    eje.set_title(titulo, fontsize=12)
    eje.set_ylabel(etiqueta)
    eje.set_xlabel("Mes")
    eje.grid(axis="y", alpha=0.25)

    eje.xaxis.set_major_locator(
        mdates.MonthLocator(interval=3)
    )
    eje.xaxis.set_major_formatter(
        mdates.DateFormatter("%m/%Y")
    )
    eje.tick_params(axis="x", rotation=45)

    if columna == "cobertura_importe_pct":
        eje.set_ylim(95, 100)
    else:
        eje.set_ylim(bottom=0)

fig.suptitle(
    "Taco Bueno · Indicadores mensuales del pipeline",
    fontsize=16
)

fig.text(
    0.5,
    0.02,
    "Las tiendas y fechas disponibles varían entre meses. "
    "Los días sin registros no se cuentan como cero.",
    ha="center",
    fontsize=10
)

plt.tight_layout(rect=[0, 0.06, 1, 0.94])
plt.show()

In [0]:
comparacion = spark.sql("""
    SELECT
        COUNT(DISTINCT store_number) AS tiendas,
        COUNT(*) AS pares_tienda_fecha,
        SUM(pedidos_2025) AS pedidos_2025,
        SUM(pedidos_2026) AS pedidos_2026
    FROM workspace.default.gold_yoy_store_day
""").first()

pares = comparacion["pares_tienda_fecha"]

if pares == 0:
    raise ValueError("No se encontraron fechas comparables.")

promedio_2025 = comparacion["pedidos_2025"] / pares
promedio_2026 = comparacion["pedidos_2026"] / pares
cambio = 100 * (promedio_2026 / promedio_2025 - 1)

print(f"Tiendas comparables: {comparacion['tiendas']:,}")
print(f"Pares de tienda y fecha: {pares:,}")
print(f"Pedidos de 2025: {comparacion['pedidos_2025']:,}")
print(f"Pedidos de 2026: {comparacion['pedidos_2026']:,}")
print(f"Pedidos por tienda-día en 2025: {promedio_2025:.2f}")
print(f"Pedidos por tienda-día en 2026: {promedio_2026:.2f}")
print(f"Cambio en pedidos por tienda-día: {cambio:+.2f}%")

In [0]:
import matplotlib.pyplot as plt

comparacion_mensual = spark.sql("""
    SELECT
        mes_numero,
        COUNT(*) AS pares,
        SUM(pedidos_2025) AS pedidos_2025,
        SUM(pedidos_2026) AS pedidos_2026
    FROM workspace.default.gold_yoy_store_day
    GROUP BY mes_numero
    ORDER BY mes_numero
""").toPandas()

comparacion_mensual["promedio_2025"] = (
    comparacion_mensual["pedidos_2025"]
    / comparacion_mensual["pares"]
)

comparacion_mensual["promedio_2026"] = (
    comparacion_mensual["pedidos_2026"]
    / comparacion_mensual["pares"]
)

comparacion_mensual["cambio_pct"] = 100 * (
    comparacion_mensual["promedio_2026"]
    / comparacion_mensual["promedio_2025"] - 1
)

nombres_meses = {
    2: "Feb", 3: "Mar", 4: "Abr", 5: "May",
    6: "Jun", 7: "Jul", 8: "Ago", 9: "Sep"
}

comparacion_mensual["mes"] = (
    comparacion_mensual["mes_numero"].map(nombres_meses)
)

fig, ejes = plt.subplots(1, 2, figsize=(14, 5))

for anio, color in [(2025, "#2563EB"), (2026, "#D97706")]:
    ejes[0].plot(
        comparacion_mensual["mes"],
        comparacion_mensual[f"promedio_{anio}"],
        marker="o",
        linewidth=2,
        label=str(anio),
        color=color
    )

ejes[0].set_title("Pedidos por tienda-día comparable")
ejes[0].set_ylabel("Pedidos")
ejes[0].set_ylim(bottom=0)
ejes[0].legend()

colores = [
    "#DC2626" if cambio < 0 else "#059669"
    for cambio in comparacion_mensual["cambio_pct"]
]

barras = ejes[1].bar(
    comparacion_mensual["mes"],
    comparacion_mensual["cambio_pct"],
    color=colores
)

ejes[1].bar_label(
    barras,
    labels=[
        f"{valor:+.1f}%"
        for valor in comparacion_mensual["cambio_pct"]
    ],
    padding=4
)

ejes[1].axhline(0, color="black", linewidth=0.8)
ejes[1].set_title("Cambio de 2026 respecto a 2025")
ejes[1].set_ylabel("Cambio porcentual")
ejes[1].margins(y=0.25)

for eje in ejes:
    eje.set_xlabel("Mes")
    eje.grid(axis="y", alpha=0.2)
    eje.set_axisbelow(True)

fig.suptitle(
    "Taco Bueno · Comparación con las mismas tiendas y fechas",
    fontsize=15
)

fig.text(
    0.5, 0.02,
    "Solo pares presentes en ambos años. Septiembre: hasta el día 27.",
    ha="center",
    fontsize=10
)

plt.tight_layout(rect=[0, 0.06, 1, 0.94])
plt.show()

display(
    comparacion_mensual[
        ["mes", "pares", "promedio_2025", "promedio_2026", "cambio_pct"]
    ].round(2)
)

### Resultado principal

Se compararon 109 tiendas usando 23,235 pares de tienda y fecha
presentes en ambos años, entre febrero y el 27 de septiembre.
Los pedidos por tienda-día pasaron de 223.65 a 211.75:
una disminución de 5.32 %. Siete de los ocho meses mostraron
disminuciones de lñs las mayores fueron julio y septiembre.

Estos resultados permiten identificar los periodos que requieren
más investigación. Los pedidos no representan clientes únicos.


In [0]:
%sql
SELECT
    COUNT(*) AS tiendas_comparadas,

    SUM(
        CASE WHEN cambio_pedidos < 0 THEN 1 ELSE 0 END
    ) AS tiendas_con_caida,

    SUM(
        CASE WHEN cambio_pedidos > 0 THEN 1 ELSE 0 END
    ) AS tiendas_con_crecimiento,

    SUM(
        CASE WHEN cambio_pedidos = 0 THEN 1 ELSE 0 END
    ) AS tiendas_sin_cambio,

    SUM(pares_tienda_fecha) AS pares_tienda_fecha,

    SUM(pedidos_2025) AS pedidos_2025,
    SUM(pedidos_2026) AS pedidos_2026,

    SUM(cambio_pedidos) AS cambio_total_pedidos

FROM workspace.default.gold_yoy_store_summary;

In [0]:
import matplotlib.pyplot as plt

# Solo llevamos a pandas el resumen de las 109 tiendas.
tiendas = spark.table(
    "workspace.default.gold_yoy_store_summary"
).toPandas()

# Seleccionar las tiendas que disminuyeron sus pedidos.
caidas = tiendas[
    tiendas["cambio_pedidos"] < 0
].copy()

caidas["pedidos_menos"] = -caidas["cambio_pedidos"]

# Pérdida acumulada de las tiendas que bajaron,
# antes de compensarla con las tiendas que crecieron.
perdida_total = int(caidas["pedidos_menos"].sum())

top10 = (
    caidas
    .sort_values("pedidos_menos", ascending=False)
    .head(10)
    .copy()
)

top10["participacion_caidas_pct"] = (
    100 * top10["pedidos_menos"] / perdida_total
)

participacion_top10 = (
    100 * top10["pedidos_menos"].sum() / perdida_total
)

print(f"Tiendas con caída: {len(caidas)}")
print(f"Pedidos menos en esas tiendas: {perdida_total:,}")
print(
    "Las diez tiendas con mayores pérdidas concentran "
    f"{participacion_top10:.2f}% de esa disminución."
)

# Mostrar el detalle.
display(
    top10[
        [
            "store_number",
            "pares_tienda_fecha",
            "pedidos_2025",
            "pedidos_2026",
            "pedidos_menos",
            "cambio_pct",
            "participacion_caidas_pct"
        ]
    ].round(2)
)

# Gráfica: las mayores pérdidas aparecen arriba.
grafica = top10.sort_values("pedidos_menos")

fig, eje = plt.subplots(figsize=(10, 6))

barras = eje.barh(
    grafica["store_number"].astype(str),
    grafica["pedidos_menos"],
    color="#DC2626"
)

eje.bar_label(
    barras,
    labels=[
        f"{int(valor):,}"
        for valor in grafica["pedidos_menos"]
    ],
    padding=5
)

eje.set_title("Diez tiendas con mayor disminución de pedidos")
eje.set_xlabel("Pedidos menos en 2026 frente a 2025")
eje.set_ylabel("Identificador de tienda")
eje.set_xlim(0, grafica["pedidos_menos"].max() * 1.20)
eje.grid(axis="x", alpha=0.2)
eje.set_axisbelow(True)

fig.text(
    0.5,
    0.02,
    "Comparación con fechas presentes en ambos años: "
    "1 de febrero a 27 de septiembre.",
    ha="center",
    fontsize=9
)

plt.tight_layout(rect=[0, 0.06, 1, 1])
plt.show()

### ¿Dónde se concentra la disminución?

La caída de pedidos se observa en 91 de las 109 tiendas comparadas
(83.49 %), por lo que está extendida entre las sucursales.

Las diez tiendas con mayores pérdidas acumulan 96,959 pedidos menos.
La tienda 5265 presenta la mayor disminución absoluta, con 18,651.

Este resultado permite priorizar las tiendas donde conviene
investigar cambios en operación, disponibilidad y demanda.